In [1]:
import os, sys
import yaml
import xarray as xr
import geopandas as gpd
import pandas as pd
from shapely.geometry import LineString, Polygon
import numpy as np

sys.path.append('/home/dnash/repos/eaton_scripps_CO_ARs/modules')
from utils import select_months_df, get_startmon_and_endmon
from plot_trajectory_maps import subset_data_to_plot, subset_gdf_to_plot
from load_trajectories import load_trajectories_based_on_region
### Imports config name from argument when submit
yaml_doc = 'config_1.yaml'
config_name = 'job_2'

In [2]:
# import configuration file for season dictionary choice
config = yaml.load(open(yaml_doc), Loader=yaml.SafeLoader)
ddict = config[config_name]

In [4]:
# Load trajectory GeoJSON data
df = gpd.read_file("/home/dnash/repos/eaton_scripps_CO_ARs/out/trajectories.geojson")
df.crs = 'EPSG:4326'
df = df.set_index(pd.to_datetime(df['start_date']))

ssn = ddict['SSN']
ARDT = ddict['ARDT']
ar = ddict['AR']
region_lst = ddict['region_lst']

print(ssn, ARDT, ar, region_lst)

subset = subset_gdf_to_plot(df, ARDT, ssn, ar, region=region_lst[0], basin=None, HUC8=None)
subset = subset.reset_index(drop=True)
# Subtract 1 from 'ar_scale' to fix error
subset['ar_scale'] = subset['ar_scale'] - 1
# Replace NaN values with 0
subset['ar_scale'] = subset['ar_scale'].fillna(0)
subset

NDJFMA tARget False ['northern_upper_CO', 'southern_upper_CO', 'rio_grande', 'eastern_CO']


,landfall_time,lat,lon,HUC8,start_date,prec,ar_scale,rutz_ar,tARget,region,geometry
0,nan,NaN,NaN,14050002,2000-02-11,11.4995,0.0,NaN,NaN,northern_upper_CO,"LINESTRING (40.45464 -108.13185, 40.33641 -108..."
1,nan,NaN,NaN,14050002,2000-12-11,13.2741,0.0,NaN,NaN,northern_upper_CO,"LINESTRING (40.45464 -108.13185, 41.08013 -108..."
2,nan,NaN,NaN,14050002,2001-11-23,12.4783,0.0,NaN,NaN,northern_upper_CO,"LINESTRING (40.45464 -108.13185, 40.35898 -108..."
3,2002-03-12 22,38.00,-122.75,14050002,2002-03-14,14.2798,1.0,1.0,NaN,northern_upper_CO,"LINESTRING (40.45464 -108.13185, 40.97919 -108..."
4,2003-04-21 13,31.75,-114.50,14050002,2003-04-24,19.3318,0.0,0.0,NaN,northern_upper_CO,"LINESTRING (40.45464 -108.13185, 40.76980 -108..."
...,...,...,...,...,...,...,...,...,...,...,...
560,2019-04-28 03,31.75,-114.50,14030001,2019-04-30,17.0020,0.0,0.0,NaN,northern_upper_CO,"LINESTRING (39.09508 -109.24819, 38.83500 -109..."
561,2021-12-08 21,32.25,-117.00,14030001,2021-12-10,19.1897,0.0,0.0,NaN,northern_upper_CO,"LINESTRING (39.09508 -109.24819, 38.68073 -109..."
562,nan,NaN,NaN,14030001,2022-11-04,13.4159,0.0,NaN,NaN,northern_upper_CO,"LINESTRING (39.09508 -109.24819, 39.32186 -109..."
563,nan,NaN,NaN,14030001,2022-12-28,11.5912,0.0,NaN,NaN,northern_upper_CO,"LINESTRING (39.09508 -109.24819, 38.75456 -109..."


In [9]:
idx = (subset['ar_scale'] == 4)
subset.loc[idx]

,landfall_time,lat,lon,HUC8,start_date,prec,ar_scale,rutz_ar,tARget,region,geometry


In [26]:
row = subset.iloc[0]

In [27]:


print(x_coords)

[-108.13185211012565, -108.190757981405, -108.29763826359847, -108.37183080080422, -108.3773228956039, -108.3940663541629, -108.43730267159782, -108.48677035929965, -108.51555791768104, -108.52663199954894, -108.53274819186511, -108.54116484545003, -108.56541596128982, -108.59129464628958, -108.62652580045015, -108.66762008539779, -108.68283044210584, -108.69551312694007, -108.70364043717198, -108.71588843614875, -108.72182266869073, -108.73743760206429, -108.75426005744315, -108.76883487755605, -108.80186927084618, -108.85788772400998, -108.93735817002623, -109.01310988948912, -109.16075247718219, -109.32744451635875, -109.5038376582422, -109.6922125728633, -109.84652560095702, -109.91177666183401, -109.99140742516701, -110.1126626062613, -110.23123130747607, -110.32331627950255, -110.39954373933539, -110.50095828868095, -110.60827732131314, -110.71173842153067, -110.80714012340415, -110.87806466568635, -110.9615203218971, -111.03907382821846, -111.13357650368316, -111.263923562098, -

In [5]:
def plot_spaghetti_maps(ax, ds, datacrs):
    '''
    Given a plot Axes and data, this returns the plot with the trajectories colored by AR scale
    
    Parameters
    ----------
    ax : 
        plot Axes on which to draw the data
    
    ds : 
        data to plot trajectories based on AR scale value
        
    Returns
    -------
    ax :
        plot Axes with trajectories
    '''

    colors = ['#F5F0E6', '#0ac1ff', '#04ff03', '#ffff03', '#ffa602', '#ff0100']
    ## Loop through AR scale values
    for k in range(1, 7):
        try:
            AR = ds.where(ds.ar_scale == k, drop=True)
            for l, HUC8 in enumerate(AR.HUC8.values):
                tmp = AR.sel(HUC8=HUC8)
                tmp = tmp.where(tmp.ar_scale == k, drop=True)
                nevents = len(tmp.start_date)
                ## LOOP THROUGH TRAJECTORIES
                for m in range(nevents):
                    data = tmp.isel(start_date=m)
                    y_lst = data.lat.values
                    x_lst = data.lon.values
                    ## check if all nan
                    test1 = np.isnan(x_lst).all()
                    test2 = np.isnan(y_lst).all()
                    if (test1 == True) & (test2 == True):
                        pass
                    else:
                        ax.plot(x_lst, y_lst, c=colors[k-1], transform=datacrs, alpha=0.2)
                        cf = ax.scatter(x_lst, y_lst, c=colors[k-1], marker='.', transform=datacrs, alpha=0.7, s=6)
        except IndexError:
            pass

    return ax

NDJFMA tARget True ['northern_upper_CO', 'southern_upper_CO', 'rio_grande', 'eastern_CO']


,landfall_time,lat,lon,HUC8,start_date,prec,ar_scale,rutz_ar,tARget,region,geometry
0,2003-10-31 08,29.25,-112.25,14050002,2003-11-03,11.6626,2.0,0.0,2.003103e+11,northern_upper_CO,"LINESTRING (40.45464 -108.13185, 40.36599 -108..."
1,2004-11-25 03,41.00,-124.00,14050002,2004-11-28,12.8497,2.0,1.0,2.004112e+11,northern_upper_CO,"LINESTRING (40.45464 -108.13185, 40.04437 -108..."
2,2005-01-09 06,31.00,-113.00,14050002,2005-01-12,15.8560,NaN,1.0,2.005011e+11,northern_upper_CO,"LINESTRING (40.45464 -108.13185, 40.10463 -108..."
3,2005-04-23 20,31.75,-114.50,14050002,2005-04-25,16.4895,2.0,0.0,2.005042e+11,northern_upper_CO,"LINESTRING (40.45464 -108.13185, 40.35044 -108..."
4,2007-11-29 17,31.25,-113.25,14050002,2007-12-01,18.3882,2.0,1.0,2.007113e+11,northern_upper_CO,"LINESTRING (40.45464 -108.13185, 39.87323 -108..."
...,...,...,...,...,...,...,...,...,...,...,...
431,2019-02-28 11,34.50,-119.75,14030001,2019-03-03,13.0519,2.0,0.0,2.019022e+11,northern_upper_CO,"LINESTRING (39.09508 -109.24819, 38.80695 -109..."
432,2019-03-10 09,28.50,-111.75,14030001,2019-03-13,14.0493,3.0,1.0,2.019031e+11,northern_upper_CO,"LINESTRING (39.09508 -109.24819, 39.00784 -109..."
433,2020-03-17 17,31.00,-113.00,14030001,2020-03-19,16.3180,NaN,0.0,2.020032e+11,northern_upper_CO,"LINESTRING (39.09508 -109.24819, 38.87922 -109..."
434,2021-03-11 12,31.25,-113.50,14030001,2021-03-14,11.8672,NaN,0.0,2.021031e+11,northern_upper_CO,"LINESTRING (39.09508 -109.24819, 39.08311 -109..."


In [ ]:
ext = [-140., -90., 20, 50]
    
# Set tick/grid locations
tx = 10
ty = 5
dx = np.arange(ext[0],ext[1]+tx,tx)
dy = np.arange(ext[2],ext[3]+ty,ty)

nrows = 1
ncols = 1

## Use gridspec to set up a plot with a series of subplots that is
## n-rows by n-columns
gs = GridSpec(nrows, ncols, height_ratios=[1], width_ratios = [1], wspace=0.01, hspace=0.1)
## use gs[rows index, columns index] to access grids

fig = plt.figure(figsize=(8, 8.))
fig.dpi = 600
path_to_figs = '/home/dnash/repos/eaton_scripps_CO_ARs/figs/trajectory_figs/'
fname = path_to_figs + 'tmp'
fmt = 'png'

ax = fig.add_subplot(gs[row,col], projection=mapcrs)
ax = draw_basemap(ax, extent=ext, xticks=dx, yticks=dy, left_lats=False, 
                  right_lats=False, bottom_lons=blon_lst[i])
ax.set_extent(ext, datacrs)
ax.add_feature(cfeature.STATES, edgecolor='0.4', linewidth=0.8)
ax = plot_spaghetti_maps(ax, subset, datacrs)